In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import math
from torch.optim.lr_scheduler import StepLR
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt
from torch.cuda.amp import autocast, GradScaler


tezheng=600
shuliang=3
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=256):
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(np.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x=x.unsqueeze(0)
        return x + self.pe[:, :x.size(1)]



import torch
import torch.nn as nn

class TransformerModel(nn.Module):
    def __init__(
        self, 
        d_model=512, 
        nhead=8, 
        num_layers=4, 
        input_dim=tezheng, 
        distance_output_dim=shuliang,  
        time_output_dim=shuliang       
    ):
        super(TransformerModel, self).__init__()
        self.input_layer = nn.Linear(input_dim, d_model)
        self.positional_encoding = PositionalEncoding(d_model)
        
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=2048,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        
        self.distance_head = nn.Linear(d_model, distance_output_dim)  # 距离预测头
        self.time_head = nn.Linear(d_model, time_output_dim)          # 时间预测头

    def forward(self, x):
        # [batch_size, seq_len, input_dim] -> [batch_size, seq_len, d_model]
        x = self.input_layer(x)
        
        # 
        x = self.positional_encoding(x)
        
        #  [batch_size, seq_len, d_model]
        x = self.transformer_encoder(x)
        
        
        distance_pred = self.distance_head(x)  # [batch_size, seq_len, distance_output_dim]
        time_pred = self.time_head(x)          # [batch_size, seq_len, time_output_dim]
        
        
        if x.shape[1] == 1: 
            output = torch.cat([distance_pred.squeeze(1), time_pred.squeeze(1)], dim=-1)
        else:
            output = torch.cat([distance_pred, time_pred], dim=-1)
        output = output.squeeze(0)
        return output


In [ ]:

class EarlyStopping:
    def __init__(self, patience=5, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_loss = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.delta:
            self.counter += 1
            print(f'EarlyStopping counter: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0
data = pd.read_csv('',header=None)
X = data.iloc[:, :tezheng].values
Y = data.iloc[:, tezheng:].values
                                                       
X_train, X_temp, Y_train, Y_temp = train_test_split(X, Y, test_size=0.4, random_state=42)  # 先分出60%训练集
X_val, X_test, Y_val, Y_test = train_test_split(X_temp, Y_temp, test_size=0.5, random_state=42)  # 剩余40%平分验证和测试


scaler_X = MinMaxScaler()
X_train = scaler_X.fit_transform(X_train)
X_val = scaler_X.transform(X_val)
X_test = scaler_X.transform(X_test)


X_train_t = torch.FloatTensor(X_train)
Y_train_t = torch.FloatTensor(Y_train)
X_val_t = torch.FloatTensor(X_val)
Y_val_t = torch.FloatTensor(Y_val)
X_test_t = torch.FloatTensor(X_test)
Y_test_t = torch.FloatTensor(Y_test)


batch_size = 128
train_dataset = TensorDataset(X_train_t, Y_train_t)
val_dataset = TensorDataset(X_val_t, Y_val_t)
test_dataset = TensorDataset(X_test_t, Y_test_t)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TransformerModel().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10)
early_stopping = EarlyStopping(patience=30, delta=0.001)
scaler = torch.amp.GradScaler() 

criterion_distance = nn.MSELoss()
criterion_time = nn.MSELoss()